In [27]:
library("GSEABase")
library('GSVA')
library(ggplot2)


In [28]:
##### load Moorman signature as R object
sign_f <- '/mnt/trcanmed/snaketree/stash/collection_signature_def.tsv'
sign <- read.table(sign_f, sep="\t", header=T)
sign <- sign[sign$Kept, ]
sign_l <- unique(sign$Annotation)

make_gene_set <- function(signature, data) {
  data <- data[data$Annotation == signature,]
  #geneset <- GeneSet(data$Gene, geneIdType=SymbolIdentifier()) # does not work? Y?
  #setName(geneset) <- signature
  #return(geneset)
  return(data$Gene)
}

signatures <- lapply(sign_l, make_gene_set, sign)
names(signatures) <- sign_l

# for ulisse's version 1.60.0 of GSEAbase
gene_sets_list <- signatures 
gsc <- GeneSetCollection(
  lapply(names(gene_sets_list), function(setName) {
    GeneSet(
      geneIds = unique(gene_sets_list[[setName]]),
      geneIdType = SymbolIdentifier(),
      setName = setName
    )
  })
)

In [3]:
gsc

GeneSetCollection
  names: CBC, colon, ..., coreHRC (8 total)
  unique identifiers: ACOT1, ACSM3, ..., B4GALNT3 (1317 total)
  types in collection:
    geneIdType: SymbolIdentifier (1 total)
    collectionType: NullCollection (1 total)

In [ ]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_N2.vs.NE/MUT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_N2.vs.NE/MUT_tmm.tsv.gz' # H_

In [ ]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_N2.vs.NE/WT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_N2.vs.NE/WT_tmm.tsv.gz' # H_

In [8]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_bcat.vs.scr/bcat_WT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/WT_bcat.vs.scr/WT_tmm.tsv.gz' # H_

In [21]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_bcat.vs.scr/bcat_MUT_samples_data'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/MUT_bcat.vs.scr/MUT_tmm.tsv.gz' # H_

In [29]:
meta_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/basali_MUT.vs.WT/mut_samples_data_NE'
tmm_f <- '/mnt/cold1/snaketree/prj/DE_RNASeq/dataset/TCF7L2_2nd/basali_MUT.vs.WT/tmm.tsv.gz' # H_

In [30]:
meta <- read.table(meta_f, sep="\t", header=T)
expr_data <- read.table(gzfile(tmm_f), sep="\t", header=TRUE, row.names=1)

#gsva_par <- gsvaParam(exprData=as.matrix(expr_w), geneSets=signatures_s,  kcdf="Gaussian") # godot
#res <- gsva(gsva_par, verbose=T)
#res <- gsva(as.matrix(expr_data), gsc, kcdf="Gaussian", 'gsva') # ulisse


In [31]:
gsva_par <- gsvaParam(exprData=as.matrix(expr_data), geneSets=gsc,  kcdf="Gaussian") # ulisse
res <- gsva(gsva_par, verbose=T)


ℹ GSVA version 2.0.0

ℹ Calculating GSVA ranks

ℹ GSVA dense (classical) algorithm

ℹ Row-wise ECDF estimation with Gaussian kernels

ℹ Calculating GSVA column ranks

! No annotation metadata available in the input expression data object

! Attempting to directly match identifiers in expression data to gene sets

ℹ Calculating GSVA scores

✔ Calculations finished



In [32]:
tres <- as.data.frame(t(res))
tres$stem_index <- tres$RSC - tres$CBC

In [33]:
head(tres)

,CBC,colon,esophagus,onf,stem,RSC,coreHRC,stem_index
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CRC0080_NE_R1,-0.12606191,0.2598612,0.3453962,0.30237702,-0.6587260,0.32143422,0.359446904,0.44749613
CRC0080_NE_R2,-0.06349167,0.2056626,0.1577388,0.16842663,-0.2339888,0.20473062,-0.136808687,0.26822229
CRC0148_NE_R1,-0.09983041,0.1728954,0.1602500,-0.03615706,-0.3916825,-0.01222547,-0.003756513,0.08760493
CRC0148_NE_R2,-0.09412660,0.1763531,0.1975740,-0.02834643,-0.3580540,-0.02868090,0.057784951,0.06544569
CRC0515_NE_R1,0.11693969,0.2005705,-0.1597548,-0.26503224,0.4112854,-0.23261091,-0.578164886,-0.34955060
CRC0515_NE_R2,0.06362056,0.1136767,-0.1182805,-0.12109229,0.4772122,-0.17219447,-0.530897658,-0.23581504


In [21]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_mut.tsv', sep="\t", quote=F)

In [13]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_bcat_wt.tsv', sep="\t", quote=F)

In [26]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_bcat_mut.tsv', sep="\t", quote=F)

In [35]:
write.table(tres, file='/mnt/trcanmed/snaketree/stash/tcf7l2_pla_basali.tsv', sep="\t", quote=F)